# SQL — Date/Time Functions and Gaps & Islands

<div style="padding: 15px; border-left: 8px solid #1a8a5a; background-color: #e6f4ea; color: #0d532a; border-radius: 4px;">
<strong>🎯 Core Insight:</strong> Time-series SQL is what separates data engineers from SQL developers. "Gaps & islands" is the ultimate pattern showing you understand how to detect missing data or consecutive blocks in sequences. A common trick is subtracting a <code>ROW_NUMBER()</code> from an incrementing date to generate a constant identifier for consecutive dates.
</div>

### 💼 The Citi Angle: Capacity Planning & Server Reporting
At Citi, managing thousands of trading nodes means constantly verifying server uptime and health reporting. Gap detection on server reporting is daily work. When capacity planning teams need to know a server's longest continuous uptime "streak" (an Island) or identify exactly which days a node silently dropped offline without throwing a hard error (a Gap), they rely on these exact time-series SQL patterns to reconstruct the timeline from fragmented daily ping metrics.

In [8]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# Your existing settings
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "practice_db"
DB_USER = "postgres"
DB_PASS = "mysecretpassword"

conn_str = f"postgresql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_str)

def run_sql(query: str):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(query))
            df = pd.DataFrame(result.mappings())   # ← this handles SQLAlchemy 2.0+
            display(df)
    except Exception as e:
        print(f"Error executing query:\n{e}")

print("✅ Fixed SQL execution environment ready!")

✅ Fixed SQL execution environment ready!


In [9]:
# Setup: Drop existing and create mock data table for daily_metrics
setup_sql = """
DROP TABLE IF EXISTS daily_metrics;

CREATE TABLE daily_metrics AS 
SELECT * FROM (
    VALUES 
    ('SRV-01', '2026-02-01'::DATE, 100),
    ('SRV-01', '2026-02-02'::DATE, 105),
    ('SRV-01', '2026-02-03'::DATE, 98),
    ('SRV-01', '2026-02-04'::DATE, 102),
    -- GAP: 2026-02-05 is missing
    ('SRV-01', '2026-02-06'::DATE, 101),
    ('SRV-01', '2026-02-07'::DATE, 99),
    -- GAP: 2026-02-08, 09, 10 are missing
    ('SRV-01', '2026-02-11'::DATE, 110),
    ('SRV-01', '2026-02-12'::DATE, 112),
    ('SRV-01', '2026-02-13'::DATE, 108),
    ('SRV-01', '2026-02-14'::DATE, 109),
    
    ('SRV-02', '2026-02-01'::DATE, 50),
    ('SRV-02', '2026-02-02'::DATE, 52),
    ('SRV-02', '2026-02-05'::DATE, 55),
    ('SRV-02', '2026-02-06'::DATE, 51)
) AS t(server_id, report_date, cpu_usage);
"""
with engine.connect() as conn:
    conn.execute(text(setup_sql))
    conn.commit()
print("✅ Mock table 'daily_metrics' created successfully with Gaps and Islands.")

✅ Mock table 'daily_metrics' created successfully with Gaps and Islands.


In [10]:
# -------------------------------------------------------------------
# Pattern 1: Core Date Functions & Filtering
# Expected Output: Shows various date extracts (yr, mo, dow) 
# and truncations (week_start, month_start), filtering out weekends.
# -------------------------------------------------------------------
sql="""
SELECT 
    server_id,
    report_date,
    EXTRACT('dow' FROM report_date) AS day_of_week,
    DATE_TRUNC('week', report_date)::DATE AS week_start,
    DATE_TRUNC('month', report_date)::DATE AS month_start
FROM daily_metrics
WHERE EXTRACT('dow' FROM report_date) NOT IN (0, 6) -- 0=Sun, 6=Sat
ORDER BY server_id, report_date
LIMIT 5;
"""
run_sql(sql)

,day_of_week,month_start,report_date,server_id,week_start
0,1,2026-02-01,2026-02-02,SRV-01,2026-02-02
1,2,2026-02-01,2026-02-03,SRV-01,2026-02-02
2,3,2026-02-01,2026-02-04,SRV-01,2026-02-02
3,5,2026-02-01,2026-02-06,SRV-01,2026-02-02
4,3,2026-02-01,2026-02-11,SRV-01,2026-02-09


In [11]:
# -------------------------------------------------------------------
# Pattern 2: Gaps & Islands — The Core Pattern
# Expected Output: Groups contiguous date blocks (Islands) by subtracting
# a row_number from the report_date to form a stable island_key.
# -------------------------------------------------------------------
sql="""
WITH numbered AS (
    SELECT
        server_id,
        report_date,
        ROW_NUMBER() OVER (PARTITION BY server_id ORDER BY report_date) AS rn,
        report_date - CAST(ROW_NUMBER() OVER (
            PARTITION BY server_id ORDER BY report_date
        ) AS INT) AS island_key     -- dates in same island share this value
    FROM daily_metrics
),
islands AS (
    SELECT
        server_id,
        island_key,
        MIN(report_date) AS island_start,
        MAX(report_date) AS island_end,
        COUNT(*) AS consecutive_days
    FROM numbered
    GROUP BY server_id, island_key
)
SELECT
    server_id,
    island_start,
    island_end,
    consecutive_days
FROM islands
ORDER BY server_id, island_start;
"""
run_sql(sql)

,consecutive_days,island_end,island_start,server_id
0,4,2026-02-04,2026-02-01,SRV-01
1,2,2026-02-07,2026-02-06,SRV-01
2,4,2026-02-14,2026-02-11,SRV-01
3,2,2026-02-02,2026-02-01,SRV-02
4,2,2026-02-06,2026-02-05,SRV-02


In [12]:
# -------------------------------------------------------------------
# Pattern 3: Consecutive Days (Streaks >= N)
# Expected Output: Filters the islands to only show servers that reported
# data for 3 or more consecutive days.
# -------------------------------------------------------------------
sql="""
WITH daily AS (
    SELECT server_id, report_date,
           report_date - CAST(ROW_NUMBER() OVER (
               PARTITION BY server_id ORDER BY report_date
           ) AS INT) AS grp
    FROM daily_metrics
)
SELECT server_id, 
       MIN(report_date) AS streak_start,
       MAX(report_date) AS streak_end,
       COUNT(*) AS streak_length
FROM daily
GROUP BY server_id, grp
HAVING COUNT(*) >= 3
ORDER BY streak_length DESC;
"""
run_sql(sql)

,server_id,streak_end,streak_length,streak_start
0,SRV-01,2026-02-14,4,2026-02-11
1,SRV-01,2026-02-04,4,2026-02-01


In [13]:
# -------------------------------------------------------------------
# Pattern 4: Finding Actual Gaps (Using generate_series)
# Expected Output: Shows exact dates where servers should have reported
# but did not within the target February range.
# -------------------------------------------------------------------
sql="""
WITH all_dates AS (
    -- Generate all expected dates in range (PostgreSQL syntax)
    SELECT generate_series(
        '2026-02-01'::DATE,
        '2026-02-14'::DATE,  -- Truncated to our mock data bounds
        INTERVAL '1 day'
)::DATE AS expected_date
),
reported AS (
    SELECT DISTINCT server_id, report_date FROM daily_metrics
),
cross_product AS (
    SELECT s.server_id, d.expected_date
    FROM (SELECT DISTINCT server_id FROM daily_metrics) s
    CROSS JOIN all_dates d
)
SELECT c.server_id, c.expected_date AS missing_date
FROM cross_product c
LEFT JOIN reported r
    ON c.server_id = r.server_id AND c.expected_date = r.report_date
WHERE r.report_date IS NULL
ORDER BY c.server_id, c.expected_date;
"""
run_sql(sql)

,missing_date,server_id
0,2026-02-05,SRV-01
1,2026-02-08,SRV-01
2,2026-02-09,SRV-01
3,2026-02-10,SRV-01
4,2026-02-03,SRV-02
5,2026-02-04,SRV-02
6,2026-02-07,SRV-02
7,2026-02-08,SRV-02
8,2026-02-09,SRV-02
9,2026-02-10,SRV-02


### 🎤 Interview Q&A

**Q1: Explain the gaps & islands "date minus row number" trick.**
A: Consecutive dates form an arithmetic sequence where `date[i] - i` is constant. If any date is missing, the sequence breaks, and `date - rn` changes. All dates in the same consecutive block share the exact same `date - rn` value.

---

**Q2: What is `generate_series()` and where does it work?**
A: It's a native PostgreSQL and DuckDB function that forcefully generates a series of dates or numbers. Snowflake equivalent: `GENERATOR(rowcount=>N)`. BigQuery: unnest a date array. SQL Server: recursive CTE. It is the core tool used to create calendar/scaffolding tables.

---

**Q3: How would you find the longest streak?**
A: Write the Islands query → add `MAX(consecutive_days)` or do a standard `ORDER BY consecutive_days DESC`. You tie-break using `island_start` ascending if multiple streaks share the same maximum duration.

---

**Q4: What are the performance implications of the gaps and islands row-number subtraction approach?**
A: It relies heavily on `ROW_NUMBER() OVER(PARTITION BY ... ORDER BY ...)`. Window functions require sorting the data partition, which is inherently an $O(N \log N)$ operation per partition. On massive datasets, this sort overhead requires strong indexing on the partition and ordering columns to avoid spilling to disk.

---

**Q5: Why do we use CROSS JOIN against a generated series of dates to find missing records?**
A: A `LEFT JOIN` requires a spine (a master set of all possibilities) on the left side. If a server completely crashed on Feb 5th, Feb 5th doesn't exist *anywhere* in that server's log. By `CROSS JOIN`ing the unique servers with the comprehensive generated Date Series, we force a complete Cartesian grid of "what *should* exist", ensuring our `LEFT JOIN ... WHERE right_side IS NULL` catches the voids.

### 🔑 Key Terms & Checklist

| Term | Definition |
|---|---|
| **Island** | A contiguous sequence of records (e.g. consecutive days of activity). |
| **Gap** | The missing intervals between Islands where no data exists. |
| **Row Number Subtraction** | The algorithmic key to Islands: `date - ROW_NUMBER()` yields a stable ID for consecutive dates. |
| **generate_series** | Postgres/DuckDB utility to dynamically spawn a contiguous series without a physical table. |
| **EXTRACT / DATE_PART** | Pulling specific integers (Year, Month, DOW) from a Timestamp/Date field. |
| **DATE_TRUNC** | Flooring a date to a specific granularity (e.g. rounding down to the start of the week). |
| **Cartesian Grid / Date Spine** | Using a Cross Join to produce every potential Server+Date combination. |
| **Anti-Join** | A `LEFT JOIN` combined with `WHERE right_id IS NULL` to find only records present on the left matching nothing on the right. |

**Canonical Pattern Snippet:**
```sql
REPORT_DATE - CAST(ROW_NUMBER() OVER (PARTITION BY server_id ORDER BY report_date) AS INT) AS island_key
```

**Checklist:**
- [ ] Did you properly cast the ROW_NUMBER output to an integer before subtracting it from the date?
- [ ] Did you partition by the correct granular entity (e.g. `server_id` or `user_id`)?
- [ ] Did you ensure your Date Spine Cross Join didn't accidentally multiply against non-distinct Server IDs?
- [ ] Are your boundary constraints strictly filtering weekends correctly (`EXTRACT(DOW)` where 0 is usually Sunday!)?
- [ ] Did you account for leap years and month-end wraparounds when using hard interval addition/subtraction?

---
*"Simplicity and clarity is Gold." — Sean's Study Mantra 🚀*